# CSPICE Validation

In [ ]:
import os
import pylupnt as pnt
import spiceypy as sp
import numpy as np

np.set_printoptions(precision=12, linewidth=100, suppress=False)

In [ ]:
kernel_dir = pnt.get_cspice_kernel_dir()
sp.furnsh(os.path.join(kernel_dir, "de440.bsp"))
sp.furnsh(os.path.join(kernel_dir, "naif0012.tls"))

## Time Conversions

In [43]:
for year in (2000, 2025, 2050):
    date = f"{year} May 21, 01:02:03 UTC"
    t_tdb = sp.str2et(date)
    for time in ("UTC", "TT", "TDB"):
        line = time.ljust(3) + " "
        line += sp.timout(t_tdb, f"YYYY-MM-DDTHR:MN:SC.###### ::{time}") + " "
        if time == "UTC":
            line += "0"
        else:
            line += str(sp.unitim(t_tdb, "TDB", time))
        print(line)

UTC 2000-05-21T01:02:03.000000 0
TT  2000-05-21T01:03:07.184000 12142987.184
TDB 2000-05-21T01:03:07.185136 12142987.185136192
UTC 2025-05-21T01:02:03.000000 0
TT  2025-05-21T01:03:12.184000 801061392.184
TDB 2025-05-21T01:03:12.185146 801061392.1851462
UTC 2050-05-21T01:02:03.000000 0
TT  2050-05-21T01:03:12.184000 1589979792.184
TDB 2050-05-21T01:03:12.185156 1589979792.185156


## Body Positions

In [128]:
t_tai

643294960.0000002

In [130]:
t_utc = pnt.gregorian2time(2020, 5, 21, 1, 2, 3.0)
t_tai = pnt.convert_time(t_utc, pnt.UTC, pnt.TAI)
t_tdb = pnt.convert_time(t_utc, pnt.UTC, pnt.TDB)
locations = ("SSB", "SUN", "EMB", "EARTH", "MOON", "MARS_BARYCENTER")
width = max(len(x) for x in locations) + 1
with open("body_pos_vel_cspice.txt", "w") as f:
    line = f"t_tai {t_tai:.16e}"
    print(line)
    f.write(line + "\n")

    # Header: from to lt x y z vx vy vz
    line = " ".join(
        x
        for x in ("center", "target", "lt [s]", "x [km]", "y [km]", "z [km]", "vx [km/s]", "vy [km/s]", "vz [km/s]")
    )
    f.write(line + "\n")
    print(line)
    for center in locations:
        for target in locations:
            if center != target:
                rv, lt = sp.spkezr(target, t_tdb, "J2000", "NONE", center)
                line = center.ljust(width) + " " + target.ljust(width) + f" {lt:.9e} "
                line += " ".join(f"{x:+20.9f}" for x in rv)
                f.write(line + "\n")
                print(line)

t_tai 6.4329496000000024e+08
center target lt [s] x [km] y [km] z [km] vx [km/s] vy [km/s] vz [km/s]
SSB              SUN              4.306222834e+00    -741987.269464739    +965929.273698082    +427840.095432443         -0.014083820         -0.005627821         -0.002001045
SSB              EMB              5.032556135e+02  -76045916.558535278 -119558495.426486567  -51819077.151945584        +25.343863212        -13.700933974         -5.938948538
SSB              EARTH            5.032710330e+02  -76049587.622088879 -119561571.417796656  -51820051.626781650        +25.351890403        -13.708691485         -5.943122540
SSB              MOON             5.020021891e+02  -75747457.005650803 -119308415.585131600  -51739851.794055827        +24.691248060        -13.070243951         -5.599599808
SSB              MARS_BARYCENTER  7.038906758e+02  +70984578.570678294 -179883009.673950434  -84458296.051201954        +23.705224294         +9.564791120         +3.747645442
SUN              SS